# Client GUI Usage

Create a Sen4CAP client, open the GUI, and edit its shared process-request state. This notebook connects to a live service; running a process in the GUI submits a real job.

In [ ]:
from sen4cap_client.api import create_client

## Create a client instance

Run `sen4cap-client configure` and `sen4cap-client login` in a terminal first.
Enter the processing and login URLs supplied by your service administrator; the
defaults point to a local server. The factories read the saved Sen4CAP profile
and keyring credentials. To use another profile, pass `config_path="./sen4cap.yaml"`.

Authentication overrides belong inside `auth`, for example
`create_client(auth={"username": "...", "password": "..."})` for a profile using
login authentication. Prefer the CLI login or environment variables over storing
credentials in a notebook. See [configuration](https://Sen4CAP.github.io/sen4cap-client/configuration/).

Create the client and log in before opening the app.

In [ ]:
client = create_client()

In [ ]:
client.login()

## Show the app

In [ ]:
# In a notebook, the default display="auto" embeds the app.
# Use display="browser" to open a separate browser tab.
app = client.show_app(height=640)

In [ ]:
process_id = "218"  # Example NDVI process; adapt to your service.
client.get_process(process_id)

In [ ]:
# Use the exact input keys from your process description; these may be UUIDs.
app.set_process_request(
    process_id,
    {
        "inputs": {
            "c30145a7-029c-4499-98bc-9903ca46531c": "2024-06-03",
            "472efeab-514a-4e15-9dba-d5812d653065": "2024-06-11",
            "691adc8e-9bba-4f42-86e2-ccd72189edc3": "NDVI",
            "bed1920e-51c0-406e-b22e-70d1f86d95d4": "POLYGON ((9.66 53.75,10.38 53.75,10.38 53.35,9.66 53.35,9.66 53.75))",
        },
        "outputs": {
            "db96d3e8-0243-48dc-ac9a-6470d2c39eb8": {
                "format": {"mediaType": "application/json"},
                "transmissionMode": "reference",
            }
        },
    },
)

In [ ]:
app.get_process_request(process_id)

In [ ]:
# Nested edits are also supported.
app.process_requests[process_id].inputs["472efeab-514a-4e15-9dba-d5812d653065"] = "2024-06-12"

## Run and inspect a job

The state edits above update the request form; they do not submit a job.
Review the form and execute it in the GUI. Copy its returned job ID to open a
result using Python. The STAC/raster URLs must be independently accessible.

In [ ]:
job_id = input("Job ID returned by the GUI: ").strip()
data_array = client.open_job_result(job_id=job_id, asset_name="SNDVI", timeout=3600)
data_array

## Clean up

When finished, close the raster, stop the app server, and close the client.

In [ ]:
data_array.close()
app.serve_result.stop()
client.close()